In [1]:
import os
os.chdir('..')

In [24]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch.multiprocessing as mp
from src.data.cwru_data import cwru_transform
from src.features.cwt import cwt2scalogram
import timm
import onnxruntime as ort
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType
import onnxruntime

In [25]:
class CWRUDataset(Dataset):
    def __init__(self, path):
        self.dataset = torch.load(path)
        self.data    = self.dataset['data']
        self.label   = self.dataset['label']

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        image = self.data[index].float() / 255.0
        label = self.label[index].long()
        return image, label

In [41]:
val_path = r"D:\Capstone\dataset\processed\CNN\CWRU\Spectrogram\val.pt"
val_loader = DataLoader(CWRUDataset(val_path), batch_size=256, shuffle=False)

C:\Users\VUONG\AppData\Local\Temp\ipykernel_2568\1590727923.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.dataset = torch.load(path)


In [42]:
pretrained = False
in_chans = 1
num_classes = 4

edgenext_xxs = timm.create_model(
        "edgenext_xx_small.in1k",
        pretrained=pretrained,
        in_chans=in_chans,
        num_classes=num_classes,
        drop_rate=0.1,
        drop_path_rate=0.1,
    )

ghostnetv3 = timm.create_model(
        "ghostnetv3_050",
        pretrained=pretrained,
        in_chans=in_chans,
        num_classes=num_classes,
    )

mobilenetv4 = timm.create_model(
    "mobilenetv4_conv_small_050",
    pretrained=pretrained,
    in_chans=in_chans,
    num_classes=num_classes,
    drop_path_rate=0.1,
)

tinynet_d = timm.create_model(
        "tinynet_d",
        pretrained=pretrained,
        in_chans=in_chans,
        num_classes=num_classes,
        drop_path_rate=0.2,
    )

In [36]:
class CalibDataReader(CalibrationDataReader):
    def __init__(self, data_loader, input_name='input'):
        self.data = []
        for images, _ in data_loader:
            for img in images:
                self.data.append({input_name: img.unsqueeze(0).numpy()})
            if len(self.data) >= 200:
                break
        self.index = 0

    def get_next(self):
        if self.index >= len(self.data):
            return None
        item = self.data[self.index]
        self.index += 1
        return item


def export_onnx_int8(model_instance, weight_path, onnx_fp32_path, onnx_int8_path, calib_loader):
    checkpoint = torch.load(weight_path, map_location='cpu')
    model_instance.load_state_dict(checkpoint['model_state_dict'])
    model_instance.eval()

    dummy = torch.randn(1, 1, 128, 128)
    torch.onnx.export(model_instance, dummy, onnx_fp32_path,
                      input_names=['input'], output_names=['output'],
                      dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
                      opset_version=17)
    print(f"FP32 exported → {onnx_fp32_path}")

    reader = CalibDataReader(calib_loader)
    quantize_static(
        model_input=onnx_fp32_path,
        model_output=onnx_int8_path,
        calibration_data_reader=reader,
        weight_type=QuantType.QInt8,
        activation_type=QuantType.QUInt8,
    )
    print(f"INT8 exported → {onnx_int8_path}")

In [48]:
root_path = r"D:\Capstone\models\ONNX\CNN\SPECTROGRAM\TinyNetD"

fp32_path = os.path.join(root_path, 'best_model_fp32.onnx')
int8_path = os.path.join(root_path, 'best_model_int8.onnx')


weight_path = r"D:\Capstone\models\PT\CNN\SPECTROGRAM\TinyNetD\best_model.pth"

export_onnx_int8(tinynet_d, weight_path, fp32_path, int8_path, val_loader)

C:\Users\VUONG\AppData\Local\Temp\ipykernel_2568\261954825.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(weight_path, map_location='cpu')


FP32 exported → D:\Capstone\models\ONNX\CNN\SPECTROGRAM\TinyNetD\best_model_fp32.onnx


INT8 exported → D:\Capstone\models\ONNX\CNN\SPECTROGRAM\TinyNetD\best_model_int8.onnx
